# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kareem5545163-art/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import io
import urllib.request

import pandas as pd
from IPython.display import display

CSV_NAME = "content_refresh_anonymized.csv"
RAW_URL = ("https://raw.githubusercontent.com/kareem5545163-art/"
           "flyrank-internship-ml/main/data/raw/" + CSV_NAME)


def find_local_csv(name=CSV_NAME):
    """Look for the CSV relative to the cwd and its parents (read-only)."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for rel in (Path("data/raw") / name, Path(name)):
            candidate = base / rel
            if candidate.is_file():
                return candidate
    return None


def load_starter_csv():
    local = find_local_csv()
    if local is not None:
        print(f"Loading local file: {local}")
        return pd.read_csv(local)
    try:
        print("Local copy not found; reading the raw CSV URL (read-only).")
        with urllib.request.urlopen(RAW_URL, timeout=60) as resp:
            return pd.read_csv(io.BytesIO(resp.read()))
    except Exception as err:
        print(f"Download failed ({err}). Falling back to a file upload.")
        try:
            from google.colab import files  # only exists in Colab
        except ImportError as exc:
            raise FileNotFoundError(
                f"Could not load {CSV_NAME}. Put it in ./data/raw/ or next to this "
                "notebook, then re-run."
            ) from exc
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file uploaded; cannot continue.")
        return pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))


df = load_starter_csv()
print("Shape:", df.shape)

REQUIRED = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "sessions_90d", "avg_position",
    "days_since_last_update", "content_age_days", "trend_direction", "trend_pct",
]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns {missing}. Check you loaded {CSV_NAME} "
                   "(44 columns) and not a prepared feature file.")

print("content_id unique per row:", df["content_id"].is_unique)
print("distinct pseudonymous clients:", df["client_id"].nunique())
print("content types:", df["content_type"].value_counts().to_dict())
print("is_declining_label already in raw file?", "is_declining_label" in df.columns)


Local copy not found; reading the raw CSV URL (read-only).
Shape: (30000, 44)
content_id unique per row: True
distinct pseudonymous clients: 32
content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}
is_declining_label already in raw file? False


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
SOURCE_COL = "trend_direction"
if SOURCE_COL not in df.columns:
    raise KeyError(f"{SOURCE_COL!r} is needed to build the proxy but is not in the data.")

df["target_proxy_is_declining"] = df[SOURCE_COL].eq("down").astype("int8")

print("trend_direction values (source of the rule):")
display(df[SOURCE_COL].value_counts(dropna=False).to_frame("rows"))

prev = df["target_proxy_is_declining"].mean()
print("Proxy class counts:")
display(df["target_proxy_is_declining"].value_counts().rename_axis("target_proxy_is_declining").to_frame("rows"))
print(f"Proxy prevalence: {prev:.1%} of pages are flagged 'down'. "
      "A random queue would hit this rate by chance, so precision@K must beat it.")

LEAKAGE_COLS = ["trend_direction", "trend_pct"]

ID_COLS = ["content_id", "client_id"]
print("Leakage columns (never features):", LEAKAGE_COLS)
print("Identifier columns (never features):", ID_COLS)


trend_direction values (source of the rule):


,rows
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


Proxy class counts:


,rows
target_proxy_is_declining,
1,16262
0,13738


Proxy prevalence: 54.2% of pages are flagged 'down'. A random queue would hit this rate by chance, so precision@K must beat it.
Leakage columns (never features): ['trend_direction', 'trend_pct']
Identifier columns (never features): ['content_id', 'client_id']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

K = 20


def precision_at_k(scores, relevant, k=K):
    scores = np.asarray(scores, dtype=float)
    relevant = np.asarray(relevant)
    if len(scores) < k:
        raise ValueError(f"Need at least {k} items, got {len(scores)}.")
    top_idx = np.argsort(-scores, kind="stable")[:k]
    return float(relevant[top_idx].mean())


eligible = df[df["impressions_90d"] >= 100]
base_rate = df["target_proxy_is_declining"].mean()
stale_p = precision_at_k(eligible["days_since_last_update"],
                         eligible["target_proxy_is_declining"].to_numpy())

rng = np.random.default_rng(0)
random_p = np.mean([
    precision_at_k(rng.random(len(eligible)), eligible["target_proxy_is_declining"].to_numpy())
    for _ in range(500)
])

print(f"K = {K}; eligible pages (impressions_90d >= 100): {len(eligible):,}")
print(f"Base rate of proxy (all pages):        {base_rate:.1%}")
print(f"Random queue precision@{K} (eligible):  {random_p:.1%}")
print(f"Stalest-first baseline precision@{K}:   {stale_p:.1%}")
print("Proxy diagnostic only: not a validated future metric, and not a model result.")

K = 20; eligible pages (impressions_90d >= 100): 22,006
Base rate of proxy (all pages):        54.2%
Random queue precision@20 (eligible):  59.7%
Stalest-first baseline precision@20:   65.0%
Proxy diagnostic only: not a validated future metric, and not a model result.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
LANE_COLS = (
    ID_COLS
    + ["content_type", "main_intent",
       "impressions_90d", "clicks_90d", "sessions_90d",
       "ctr", "avg_position", "days_since_last_update", "content_age_days"]
    + LEAKAGE_COLS
    + ["target_proxy_is_declining"]
)
absent = [c for c in LANE_COLS if c not in df.columns]
if absent:
    raise KeyError(f"Columns not in data: {absent}")

lane_df = df[LANE_COLS].copy()

print("GRAIN: one row = one pseudonymized content page")
print("lane_df shape:", lane_df.shape)
print("Row count matches source:", len(lane_df) == len(df))
print("Unique pages:", lane_df["content_id"].nunique(), "| clients:", lane_df["client_id"].nunique())
print("Proxy available for every row:", lane_df["target_proxy_is_declining"].notna().all())
print("avg_position == 0 (no position data):", int((lane_df["avg_position"] == 0).sum()), "rows")

CANDIDATE_FEATURES = [c for c in LANE_COLS
                      if c not in ID_COLS + LEAKAGE_COLS + ["target_proxy_is_declining"]]
print("Possible predictors (no IDs, no leakage columns):", CANDIDATE_FEATURES)

display(lane_df.head(10))
print("Proxy prevalence:")
display(lane_df["target_proxy_is_declining"].value_counts(normalize=True).rename("share").to_frame())


GRAIN: one row = one pseudonymized content page
lane_df shape: (30000, 14)
Row count matches source: True
Unique pages: 30000 | clients: 32
Proxy available for every row: True
avg_position == 0 (no position data): 1205 rows
Possible predictors (no IDs, no leakage columns): ['content_type', 'main_intent', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'content_age_days']


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,days_since_last_update,content_age_days,trend_direction,trend_pct,target_proxy_is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,0.76,10.6,20,187,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,0.05,20.3,25,445,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,0.09,36.5,20,141,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,0.49,6.2,22,463,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,0.13,44.0,14,263,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,0.03,8.5,20,147,down,-38.9,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,0.00,7.0,20,90,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,0.06,21.2,22,445,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,68,0.09,46.0,20,90,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,3,0.16,4.9,104,257,down,-29.2,1


Proxy prevalence:


,share
target_proxy_is_declining,
1,0.542067
0,0.457933


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal_cols = ["search_volume", "word_count", "scroll_rate", "ctr"]
signal_cols = [c for c in signal_cols if c in df.columns]

miss = df.groupby("content_type")[signal_cols].apply(lambda g: g.isna().mean())
print("Share of missing values by content_type (should not be filled blindly with 0):")
display(miss.round(3))

indicators = pd.DataFrame({
    "has_keyword_data": df["search_volume"].notna() if "search_volume" in df.columns else False,
    "has_word_count": df["word_count"].notna() if "word_count" in df.columns else False,
    "has_position_data": df["avg_position"].ne(0),
})
display(indicators.groupby(df["content_type"]).mean().round(3))

by_type = df.groupby("content_type")["target_proxy_is_declining"].agg(["mean", "size"])
print("Proxy rate by content_type (descriptive, not causal):")
display(by_type.round(3))
if "freshness_tier" in df.columns:
    by_fresh = df.groupby("freshness_tier")["target_proxy_is_declining"].agg(["mean", "size"])
    print("Proxy rate by freshness_tier (descriptive, not causal):")
    display(by_fresh.round(3))

Share of missing values by content_type (should not be filled blindly with 0):


,search_volume,word_count,scroll_rate,ctr
content_type,,,,
comparison article,0.000,0.000,0.003,0.0
feedly article,1.000,0.000,0.000,0.0
keyword article,0.014,0.283,0.004,0.0


,has_keyword_data,has_word_count,has_position_data
content_type,,,
comparison article,1.000,1.000,1.000
feedly article,0.000,1.000,0.652
keyword article,0.986,0.717,0.983


Proxy rate by content_type (descriptive, not causal):


,mean,size
content_type,,
comparison article,0.572,697
feedly article,0.287,2096
keyword article,0.561,27207


Proxy rate by freshness_tier (descriptive, not causal):


,mean,size
freshness_tier,,
0-30,0.511,20480
181+,0.471,174
31-90,0.589,175
91-180,0.611,9171


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.